In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import os

import matplotlib.pyplot as plt
import seaborn as sns

# Read in library & donor metadata

In [2]:
metadata_dir = '/rfs/project/rfs-iCNyzSAaucw/kk837/notebooks/CellTalkHHD/VisiumHD/metadata'

In [3]:
library_meta = pd.read_csv(f'{metadata_dir}/CellTalkHHD_VisiumHD_Library_metadata.csv')

# select based on "Publication"
library_meta = library_meta[library_meta['Publication'].isin(['Foetal-atlas_ver1'])]
# select based on "chemistry version"
library_meta = library_meta[library_meta['chemistry version']=='VisiumHD FFPE CytAssist']
# rename "Donor_ID"
library_meta.rename(columns={'Donor_ID':'per-frame_donorIDs'},inplace=True)
library_meta[['Publication','Library_ID','per-frame_donorIDs','SLX_number','Visium Slide ID']]

,Publication,Library_ID,per-frame_donorIDs,SLX_number,Visium Slide ID
0,Foetal-atlas_ver1,HEA_FOET14880396,C194,na,NaN
13,Foetal-atlas_ver1,CellTalkHHD_Human_13_BRC2757-2758-7,"BRC2757,BRC2758",NaN,H1-4CJYQ2V


In [4]:
donor_meta = pd.read_csv(f'{metadata_dir}/CellTalkHHD_VisiumHD_Donor_metadata.csv')
# rename "Donor_ID"
donor_meta.rename(columns={'Donor_ID':'donor'},inplace=True)

# Read in post-bin2cell data

In [5]:
# get list of libraries
library_list = list(library_meta['Library_ID'])
library_list

['HEA_FOET14880396', 'CellTalkHHD_Human_13_BRC2757-2758-7']

In [6]:
adatas = []
for i,library_id in enumerate(library_list):
    print(library_id)
    path_adata = f"/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/CellTalkHHD/VisiumHD/bin2cell_objects/{library_id}_b2c_cells.h5ad"
    adatas.append(sc.read_h5ad(path_adata))
    # add library id
    adatas[i].obs['library'] = library_id
    # modify barcodes
    adatas[i].obs_names = library_id + '___' + adatas[i].obs_names
    print(adatas[i].shape)
    print(adatas[i].X.data[:10]) # checking whether the counts are integer (no destripping)
    
    # for the library 'CellTalkHHD_Human_13_BRC2757-2758-7'
    # rename .uns['spatial'] key name
    if library_id=='CellTalkHHD_Human_13_BRC2757-2758-7':
        adatas[i].uns['spatial']['CellTalkHHD_Human_13_BRC2757-2758-7'] = adatas[i].uns['spatial'].pop('TYSER_01HEARTTYSERFFPE1_VHD')
        print(adatas[i].uns['spatial'].keys())
    
    print('')

HEA_FOET14880396
(163380, 18085)
[1. 1. 1. 1. 1. 1. 1. 2. 1. 1.]

CellTalkHHD_Human_13_BRC2757-2758-7
(120253, 18085)
[1. 1. 1. 1. 2. 1. 1. 1. 1. 1.]
dict_keys(['CellTalkHHD_Human_13_BRC2757-2758-7'])



# Concatenate

In [7]:
adata = adatas[0].concatenate(adatas[1:], 
                              uns_merge="unique",
                              index_unique = None, # already unique
                              batch_key=None # already in "sample" column
                             )

/tmp/ipykernel_671215/112233195.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adatas[0].concatenate(adatas[1:],


In [8]:
pd.crosstab(adata.obs['library'],adata.obs['labels_joint_source'])

labels_joint_source,primary,secondary
library,,
CellTalkHHD_Human_13_BRC2757-2758-7,72424,47829
HEA_FOET14880396,124392,38988


In [9]:
adata.obs.head()

,object_id,bin_count,array_row,array_col,labels_joint_source,in_tissue_manual,library,donor_section_ID
HEA_FOET14880396___1,1,31,2932.612903,334.000000,primary,tissue,HEA_FOET14880396,NaN
HEA_FOET14880396___2,2,26,2959.346154,336.923077,primary,tissue,HEA_FOET14880396,NaN
HEA_FOET14880396___3,3,32,2882.562500,334.187500,primary,tissue,HEA_FOET14880396,NaN
HEA_FOET14880396___4,4,28,2862.250000,515.178571,primary,tissue,HEA_FOET14880396,NaN
HEA_FOET14880396___5,5,35,2870.714286,305.171429,primary,tissue,HEA_FOET14880396,NaN


In [10]:
adata

AnnData object with n_obs × n_vars = 283633 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial', 'spatial_cropped_150_buffer'

# Add information to .obs

In [11]:
# get obs dataframe
obs = adata.obs.reset_index().copy()

### Add library metadata

In [12]:
# merge library metadata
obs = obs.merge(library_meta[['Library_ID','per-frame_donorIDs']],
          how='left',left_on='library',right_on='Library_ID')
obs = obs.drop(['Library_ID'],axis=1)
obs.head()

,index,object_id,bin_count,array_row,array_col,labels_joint_source,in_tissue_manual,library,donor_section_ID,per-frame_donorIDs
0,HEA_FOET14880396___1,1,31,2932.612903,334.000000,primary,tissue,HEA_FOET14880396,NaN,C194
1,HEA_FOET14880396___2,2,26,2959.346154,336.923077,primary,tissue,HEA_FOET14880396,NaN,C194
2,HEA_FOET14880396___3,3,32,2882.562500,334.187500,primary,tissue,HEA_FOET14880396,NaN,C194
3,HEA_FOET14880396___4,4,28,2862.250000,515.178571,primary,tissue,HEA_FOET14880396,NaN,C194
4,HEA_FOET14880396___5,5,35,2870.714286,305.171429,primary,tissue,HEA_FOET14880396,NaN,C194


### Demultiplex donors

In [13]:
# Demultiplex donor tissues which were in a frame
# Manually separated using loupe software: "donor_section_ID"

# cells which has donor_section_ID (manually assigned using loupe): multiple sections in a frame
mask = obs['donor_section_ID'].isna()==False 
obsnames1 = obs.index[mask]
# cells which are either from outside tissue or from only a very small portion of a heart (decided to be ignored)
mask = obs['donor_section_ID']=='nan' 
obsnames2 = obs.index[mask]
# cells which came from a frame with single section
mask = obs['donor_section_ID'].isna() 
obsnames3 = obs.index[mask]

obs.loc[obsnames1,'donor'] = [x.split('__')[0] for x in obs.loc[obsnames1,'donor_section_ID']]
obs.loc[obsnames2,'donor'] = 'nan'
obs.loc[obsnames3,'donor'] = obs.loc[obsnames3,'per-frame_donorIDs']
set(obs['donor'])

{'BRC2757', 'BRC2758', 'C194', 'nan'}

### Add donor metadata

In [14]:
obs = obs.merge(donor_meta,how='left',on='donor')
obs

,index,object_id,bin_count,array_row,array_col,labels_joint_source,in_tissue_manual,library,donor_section_ID,per-frame_donorIDs,donor,CS,est_CS,GA,PCW
0,HEA_FOET14880396___1,1,31,2932.612903,334.000000,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d
1,HEA_FOET14880396___2,2,26,2959.346154,336.923077,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d
2,HEA_FOET14880396___3,3,32,2882.562500,334.187500,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d
3,HEA_FOET14880396___4,4,28,2862.250000,515.178571,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d
4,HEA_FOET14880396___5,5,35,2870.714286,305.171429,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283628,CellTalkHHD_Human_13_BRC2757-2758-7___289282,289282,1,1064.000000,2945.000000,secondary,no_tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d
283629,CellTalkHHD_Human_13_BRC2757-2758-7___289286,289286,6,957.166667,2874.000000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d
283630,CellTalkHHD_Human_13_BRC2757-2758-7___289295,289295,1,940.000000,2928.000000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d
283631,CellTalkHHD_Human_13_BRC2757-2758-7___289304,289304,2,1092.000000,3007.500000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d


In [15]:
mask = ['tissue-folded' in x if x==x else False for x in obs['donor_section_ID']]
obs[mask]['library'].value_counts()

Series([], Name: count, dtype: int64)

### Create section ID

In [16]:
mask1 = obs['donor_section_ID'].isna()==False
mask2 = obs['donor_section_ID']!='nan'
mask3 = obs['donor_section_ID'].isna() 

obsnames_multi = obs.index[mask1&mask2] # multiple sections in a frame
obsnames_single = obs.index[mask3] # single section in a frame

# single
obs.loc[obsnames_single,'section_ID'] = obs.loc[obsnames_single,'library'].astype(str)+'__'+\
                                        obs.loc[obsnames_single,'donor'].astype(str)+'__1'
# multiple
obs.loc[obsnames_multi,'section_ID'] = obs.loc[obsnames_multi,"library"].astype(str)+"__"+\
                                      obs.loc[obsnames_multi,"donor_section_ID"].astype(str)
obs['section_ID'].value_counts()

section_ID
HEA_FOET14880396__C194__1                          163380
CellTalkHHD_Human_13_BRC2757-2758-7__BRC2757__2     34426
CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758__2     29696
CellTalkHHD_Human_13_BRC2757-2758-7__BRC2757__1     29618
CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758__1     26430
Name: count, dtype: int64

In [17]:
obs['section_ID'].isna().sum()

83

In [18]:
obs[obs['section_ID'].isna()]['donor'].value_counts()

donor
nan    83
Name: count, dtype: int64

In [19]:
set(obs[obs['donor']=='nan']['section_ID'])

{nan}

--> 'section_ID'==NaN are no donor assined cells (outside tissue or very small portion of a heart)<br>
--> these will be removed in the next step

### Add to adata.obs

In [20]:
adata.obs = obs.set_index('index').reindex(adata.obs_names).copy()
adata.obs

,object_id,bin_count,array_row,array_col,labels_joint_source,in_tissue_manual,library,donor_section_ID,per-frame_donorIDs,donor,CS,est_CS,GA,PCW,section_ID
HEA_FOET14880396___1,1,31,2932.612903,334.000000,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d,HEA_FOET14880396__C194__1
HEA_FOET14880396___2,2,26,2959.346154,336.923077,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d,HEA_FOET14880396__C194__1
HEA_FOET14880396___3,3,32,2882.562500,334.187500,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d,HEA_FOET14880396__C194__1
HEA_FOET14880396___4,4,28,2862.250000,515.178571,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d,HEA_FOET14880396__C194__1
HEA_FOET14880396___5,5,35,2870.714286,305.171429,primary,tissue,HEA_FOET14880396,NaN,C194,C194,NaN,NaN,11w0d,9w0d,HEA_FOET14880396__C194__1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CellTalkHHD_Human_13_BRC2757-2758-7___289282,289282,1,1064.000000,2945.000000,secondary,no_tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758__2
CellTalkHHD_Human_13_BRC2757-2758-7___289286,289286,6,957.166667,2874.000000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758__2
CellTalkHHD_Human_13_BRC2757-2758-7___289295,289295,1,940.000000,2928.000000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758__2
CellTalkHHD_Human_13_BRC2757-2758-7___289304,289304,2,1092.000000,3007.500000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,15,16,7w1d,5w1d,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758__2


# Clean up data

In [21]:
# remove donor==nan 
# which means removing section_ID==np.nan
print(adata.shape)
adata = adata[adata.obs['section_ID'].isna()==False]
print(adata.shape)

(283633, 18085)
(283550, 18085)


In [22]:
# select cells in_tissue
adata = adata[adata.obs['in_tissue_manual']=='tissue']
adata.shape

(269314, 18085)

# Remove cells in folded tissues

In [23]:
# check
mask = ['tissue-folded' in x if x==x else False for x in adata.obs['donor_section_ID']]
print(adata.obs[mask]['library'].value_counts())

# remove
adata = adata[[x==False for x in mask]]
adata.shape

Series([], Name: count, dtype: int64)


(269314, 18085)

# QC metrices

In [24]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
adata

/tmp/ipykernel_671215/86063382.py:1: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["mt"] = adata.var_names.str.startswith("MT-")


AnnData object with n_obs × n_vars = 269314 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'spatial'
    obsm: 'spatial', 'spatial_cropped_150_buffer'

In [25]:
adata.var.head()

,gene_ids,feature_types,genome,mt,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts
SAMD11,ENSG00000187634,Gene Expression,GRCh38,False,13371,0.061222,0.059421,95.035163,16488.0,9.710449
NOC2L,ENSG00000188976,Gene Expression,GRCh38,False,17016,0.067152,0.064994,93.681725,18085.0,9.802894
KLHL17,ENSG00000187961,Gene Expression,GRCh38,False,3418,0.013000,0.012916,98.730849,3501.0,8.161090
PLEKHN1,ENSG00000187583,Gene Expression,GRCh38,False,113,0.000420,0.000419,99.958042,113.0,4.736198
PERM1,ENSG00000187642,Gene Expression,GRCh38,False,139,0.000516,0.000516,99.948387,139.0,4.941642


# Save

In [26]:
adata.write("/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025/all_b2c_cells_raw.h5ad")

In [27]:
adata.X.data[:5]

array([1., 1., 1., 1., 1.], dtype=float32)